# 🈶 مترجم المانجا العربي + بوت ديسكورد — بيئة اختبار Kaggle

يجرّب كل شيء بالترتيب: **التنظيف ← الاستخراج ← TypeR ← الترجمة (Ollama)** ثم يشغّل **بوت ديسكورد** مع PostgreSQL.

**قبل البدء (في الشريط الجانبي):**
1. `Settings → Internet → On` (إلزامي: تنزيل الموديلات وOllama).
2. `Accelerator → GPU T4` (اختياري لكنه يسرّع Ollama كثيراً).
3. `Add-ons → Secrets` وأضف (كلها اختيارية ما عدا الأخير عند تشغيل البوت):
   - `GITHUB_TOKEN` — فقط إن كان الريبو خاصاً.
   - `DISCORD_TOKEN` — توكن البوت (من Discord Developer Portal).

ثم شغّل الخلايا بالترتيب. الخلايا ١–٤ إعداد لمرة واحدة، و٥ اختبار منطق البوت، و٦–١٠ تجارب على صفحاتك، و١١ تشغيل البوت.

In [ ]:
# ١) ⚙️ الإعدادات — عدّل هنا فقط
REPO_URL     = "https://github.com/Nader-Geek/Manga-AutoTranslate--Arabic"
BRANCH       = "new-engine"
OLLAMA_MODEL = "qwen2.5:7b-instruct"   # جرّب qwen2.5:14b-instruct لو الـ GPU يسمح
INPUT_PATH   = ""                      # صورة/ZIP/PDF لتجربتها. فارغ = يبحث في /kaggle/input ثم يولّد صفحة تجريبية
START_BOT    = True                    # شغّل بوت ديسكورد في الخلية الأخيرة (يحتاج سر DISCORD_TOKEN)
ADMIN_USER_IDS    = ""                 # اختياري: ids مفصولة بفاصلة (يقدرون يلغون مهام غيرهم)
ALLOWED_GUILD_IDS = ""                 # اختياري: لتقييد البوت بسيرفرات معيّنة
MAX_UPLOAD_MB    = 100                 # أقصى حجم لملف مرفوع للبوت (ديسكورد نفسه: 10MB بدون Nitro)
EXTERNAL_HOST    = "gofile"            # النتائج الأكبر من حد ديسكورد تُرفع على gofile ويُرسل رابط ("" = إيقاف)
PURGE_OLD_COMMANDS = True              # احذف أوامر السيرفرات القديمة (من بوت قديم لنفس التطبيق) عند التشغيل

In [ ]:
# ٢) 📥 جلب المشروع (من GitHub، أو من Dataset مرفوع على Kaggle يحتوي manga.py)
import base64, glob, os, shutil, subprocess, sys, time
from pathlib import Path

REPO = Path("/kaggle/working/manga")

def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name, "")

if (REPO / "manga.py").exists():
    print("المشروع موجود بالفعل:", REPO)
else:
    local = [Path(p).parent for p in glob.glob("/kaggle/input/**/manga.py", recursive=True)]
    if local:
        shutil.copytree(local[0], REPO)
        print("تم النسخ من Dataset:", local[0])
    else:
        tok = get_secret("GITHUB_TOKEN")
        cmd = ["git"]
        if tok:   # التوكن يُرسل كـ header ولا يظهر في الأوامر/المخرجات
            basic = base64.b64encode(f"x-access-token:{tok}".encode()).decode()
            cmd += ["-c", f"http.extraHeader=Authorization: Basic {basic}"]
        cmd += ["clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO)]
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode:
            err = (r.stderr or "").replace(tok, "***") if tok else (r.stderr or "")
            raise SystemExit("فشل git clone:\n" + err +
                             "\nلو الريبو خاص أضف سر GITHUB_TOKEN، أو ارفع المشروع كـ Dataset.")
        print("تم الـ clone:", BRANCH)
sys.path.insert(0, str(REPO))
os.chdir(REPO)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True, cwd=REPO).stdout)

In [ ]:
# ٣) 📦 تثبيت الاعتماديات + "تسخين" manga.py (ينصّب ما ينقصه مرة واحدة بدل أول مهمة)
t0 = time.time()
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", str(REPO / "requirements.txt"),
                    "-r", str(REPO / "bot" / "requirements.txt"),
                    "pixeltable-pgserver"], capture_output=True, text=True)
if r.returncode:   # اعرض الخطأ الحقيقي بدل إخفائه
    raise SystemExit("فشل pip:\n" + (r.stdout + r.stderr)[-2500:])
print(f"pip خلص في {time.time()-t0:.0f}ث — تسخين manga.py (قد يأخذ عدة دقائق أول مرة)...")
r = subprocess.run([sys.executable, str(REPO / "manga.py"), "--help"], capture_output=True, text=True, timeout=1800)
print("manga.py --help:", "OK" if r.returncode == 0 else f"فشل ({r.returncode})\n{(r.stdout + r.stderr)[-1500:]}")
print(subprocess.run("nvidia-smi -L || echo 'لا يوجد GPU (سيعمل Ollama على CPU وببطء)'", shell=True, capture_output=True, text=True).stdout)

In [ ]:
# ٤) 🤖 تثبيت Ollama وتشغيله وتنزيل النموذج
import urllib.request

def ollama_up():
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        return True
    except Exception:
        return False

if shutil.which("ollama") is None:
    subprocess.run("apt-get install -y -qq zstd >/dev/null 2>&1; curl -fsSL https://ollama.com/install.sh | sh",
                   shell=True)
assert shutil.which("ollama"), "تعذّر تثبيت Ollama — تأكد أن Internet مفعّل في Settings"

OLLAMA_ENV = {**os.environ, "OLLAMA_HOST": "127.0.0.1:11434", "OLLAMA_MODELS": "/tmp/ollama_models",
              "OLLAMA_CONTEXT_LENGTH": "8192", "OLLAMA_KEEP_ALIVE": "30m"}
if not ollama_up():
    OLLAMA_PROC = subprocess.Popen(["ollama", "serve"], env=OLLAMA_ENV,
                                   stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(60):
        if ollama_up():
            break
        time.sleep(1)
assert ollama_up(), "Ollama لم يبدأ — راجع /tmp/ollama.log:\n" + open("/tmp/ollama.log").read()[-1500:]
subprocess.run(["ollama", "pull", OLLAMA_MODEL], env=OLLAMA_ENV, check=True)
print(subprocess.run(["ollama", "list"], env=OLLAMA_ENV, capture_output=True, text=True).stdout)

In [ ]:
# ٥) ✅ اختبارات منطق البوت (طابور PostgreSQL، الأوامر الأربعة، الإحصائيات، الأمان)
#    تستخدم manga.py "وهمي" فلا تحتاج موديلات؛ تثبت أن كل شيء حول manga.py سليم.
from bot import devpg
pg_test = devpg.start("/tmp/mangabot_pg_test")
env = {**os.environ, "TEST_DATABASE_URL": pg_test.dsn}
ok_all = True
for name in ["test_bot_stats", "test_bot_runner", "test_bot_localrun", "test_bot_worker", "test_bot_ui"]:
    r = subprocess.run([sys.executable, "-I", f"tests/{name}.py"], cwd=REPO, env=env,
                       capture_output=True, text=True, timeout=900)
    ok = r.returncode == 0
    ok_all &= ok
    print("✅" if ok else "❌", name)
    if not ok:
        print((r.stdout + r.stderr)[-1500:])
pg_test.stop()
print("\nالنتيجة:", "كل الاختبارات نجحت" if ok_all else "فيه اختبارات فشلت")

In [ ]:
# ٦) 🖼️ اختيار صفحة للتجربة (ارفع صفحاتك كـ Dataset أو حدّد INPUT_PATH)
from PIL import Image, ImageDraw, ImageFont

def _inside_project(p):
    return any((a / "manga.py").exists() for a in Path(p).parents if str(a).startswith("/kaggle/input"))

def find_input():
    if INPUT_PATH:
        return Path(INPUT_PATH)
    exts = {".zip", ".jpg", ".jpeg", ".png", ".webp", ".pdf"}
    for p in sorted(glob.glob("/kaggle/input/**/*", recursive=True)):
        if Path(p).suffix.lower() in exts and not _inside_project(p):
            return Path(p)
    # لا يوجد شيء: ولّد صفحة تجريبية بسيطة (للتأكد من سلامة الخط فقط؛ الكشف الحقيقي يحتاج صفحة مانجا)
    img = Image.new("RGB", (800, 1100), (190, 200, 215))
    d = ImageDraw.Draw(img)
    d.ellipse((150, 200, 650, 520), fill="white", outline="black", width=5)
    try:
        f = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 44)
    except Exception:
        f = ImageFont.load_default()
    d.multiline_text((400, 360), "I WILL NEVER\nGIVE UP!", fill="black", font=f, anchor="mm", align="center")
    out = Path("/kaggle/working/sample_page.png")
    img.save(out)
    print("⚠️ لم أجد صفحات في /kaggle/input — ولّدت صفحة تجريبية. للتقييم الحقيقي ارفع صفحة مانجا فعلية.")
    return out

SRC = find_input()
print("سأجرّب على:", SRC)

In [ ]:
# ٧) 🧹 أمر /clean  — تنظيف النص من الفقاعات
os.environ.update({"OLLAMA_MODEL": OLLAMA_MODEL, "MANGA_PYTHON": sys.executable, "BOT_WORK_DIR": "/tmp/bot_work"})
from IPython.display import display
from bot.local_run import run_job, side_by_side

show = lambda lp: print(f"\r📄 صفحات: {lp.pages_done}  💬 فقاعات: {lp.bubbles_seen}", end="")
res_clean = run_job("clean", SRC, {"lama": False, "img_format": "jpg", "ocr": "en"}, on_progress=show)
print("\n" + res_clean.summary())
display(side_by_side(res_clean))

In [ ]:
# ٨) 🔎 أمر /extract — استخراج النصوص الأصلية (extract.txt + extract.json)
res_ext = run_job("extract", SRC, {"ocr": "en", "reading_order": "rtl"}, on_progress=show)
print("\n" + res_ext.summary())
print((res_ext.workdir / "extract" / "extract.txt").read_text(encoding="utf-8")[:2500])

In [ ]:
# ٩) 🧩 أمر /typer — تنظيف + إحداثيات الفقاعات (ملف typer.zip للاستيراد في TypeR Web)
import zipfile
res_typer = run_job("typer", SRC, {"lama": False, "ocr": "en"}, on_progress=show)
print("\n" + res_typer.summary())
print(zipfile.ZipFile(res_typer.files[0]).namelist()[:10])

In [ ]:
# ١٠) 🌐 أمر /translate — ترجمة عربية كاملة (Ollama) مع المسح وكتابة النص
res_tr = run_job("translate", SRC, {"dialect": "msa", "ocr": "en", "reading_order": "rtl",
                                    "format": "zip", "lama": False, "model": OLLAMA_MODEL},
                 on_progress=show)
print("\n" + res_tr.summary())
display(side_by_side(res_tr))
print("سجل التشغيل:", res_tr.log_path)

In [ ]:
# ١١) 🤖 تشغيل بوت ديسكورد (PostgreSQL مضمّن + Ollama أعلاه). اترك الخلية شغّالة.
BOT = None
token = get_secret("DISCORD_TOKEN")
if not START_BOT:
    print("START_BOT=False — تخطّينا البوت.")
elif not token:
    print("أضف سر DISCORD_TOKEN من Add-ons → Secrets ثم أعد تشغيل هذه الخلية.")
else:
    pg = devpg.start("/tmp/mangabot_pg")
    bot_env = {**os.environ, "DISCORD_TOKEN": token, "DATABASE_URL": pg.dsn,
               "OLLAMA_URL": "http://127.0.0.1:11434", "OLLAMA_MODEL": OLLAMA_MODEL,
               "MANGA_PYTHON": sys.executable, "BOT_WORK_DIR": "/tmp/bot_work",
               "ADMIN_USER_IDS": ADMIN_USER_IDS, "ALLOWED_GUILD_IDS": ALLOWED_GUILD_IDS,
               "MAX_UPLOAD_MB": str(MAX_UPLOAD_MB), "PURGE_OLD_COMMANDS": "1" if PURGE_OLD_COMMANDS else "",
               "EXTERNAL_HOST": EXTERNAL_HOST}
    BOT = subprocess.Popen([sys.executable, "-m", "bot"], cwd=REPO, env=bot_env,
                           stdout=open("/tmp/bot.log", "w"), stderr=subprocess.STDOUT)
    time.sleep(20)
    print(open("/tmp/bot.log", encoding="utf-8", errors="replace").read()[-1500:])
    if BOT.poll() is not None:
        print("❌ البوت توقف، كود الخروج:", BOT.returncode)
    else:
        print("✅ البوت يعمل — جرّب /ping ثم /clean في سيرفرك. (أبقِ الخلية التالية شغّالة)")

In [ ]:
# ١٢) 🔁 إبقاء الجلسة حيّة ومتابعة سجل البوت (اضغط Stop للإيقاف)
if BOT is not None:
    try:
        n = 0
        while BOT.poll() is None:
            time.sleep(60); n += 1
            if n % 10 == 0:
                print(open("/tmp/bot.log", encoding="utf-8", errors="replace").read()[-600:])
    except KeyboardInterrupt:
        BOT.terminate()
    print("توقف البوت، كود الخروج:", BOT.returncode)

## 🛠️ حل المشاكل

| المشكلة | الحل |
|---|---|
| `git clone` فشل | الريبو خاص → أضف سر `GITHUB_TOKEN`، أو ارفع مجلد المشروع كـ Dataset |
| تنزيل الموديلات (HuggingFace) فشل | تأكد أن Internet مفعّل |
| Ollama بطيء جداً | فعّل GPU، أو استخدم نموذجاً أصغر (`qwen2.5:3b-instruct`) |
| البوت يعمل لكن الأوامر لا تظهر | انتظر دقيقة (مزامنة الأوامر)، وتأكد أنك دعوت البوت بـ scope `applications.commands` |
| ديسكورد يرفض الاتصال/429 | أحياناً يحجب ديسكورد عناوين الخوادم المشتركة (Kaggle/Colab) — جرّب لاحقاً أو شغّل البوت على جهازك |
| الجلسة تنتهي | Kaggle يوقف الجلسات الخاملة؛ البوت هنا للتجربة وليس للاستضافة الدائمة |